# Primary-vertex $v_z$ reweighting: MC versus data

Derive the longitudinal primary-vertex ($v_z$) reweighting function for the pPb MC samples from the stored $v_z$ histograms of `processForestSimple.C`, validate it, and export it for the producer. The notebook only reads histograms and forests. Since 2026-11-27 `processForestSimple.C` uses the functions derived here, one per generator and beam direction (`vzWeightParameters`), instead of the single legacy `fVzWeight`. Rerun this notebook and update those parameters together.

**Production convention (verified in the source, not assumed).**

| Item | Evidence |
|---|---|
| `hVzUnweighted`: 60 uniform 0.5 cm bins in $[-15,15]$ cm, unit weight, filled **after** `isGoodEvent` and the overweight-event rejection, i.e. exactly the events that receive the MC weight | `processForestSimple.C:1041`, fill at `:4337` |
| Before the fill, MC flips $v_z\to-v_z$ when `isPbGoing`; data flips when `!isPbGoing` | `:4299-4304` |
| The pPb MC event weight **divides** by the function: `vzWeight = 1./fVzWeight.Eval(vz)` | `eventWeight`, `:769` |
| pPb and pp selections require $|v_z|\le15$ cm, so `fVzWeight` is never evaluated outside the histogram range | `:2377`, `:2379` |
| pp MC (CP5): stored generator weight times the CP5 $v_z$ weight $1/f$ (since 2026-11-27) | `eventWeight`, `vzWeightParameters` |
| The CP5 runner passes direction 1 (`isPbGoing`), so CP5 $v_z$ is flipped like Pb-going MC | `README.md`, "Local pp CP5 processing" |

Consequently the function to put in `fVzWeight` is the fitted **MC/Data** shape ratio $f(v_z)$, and the multiplicative MC weight it produces is
$$w_\mathrm{MC}(v_z)=\frac{1}{f(v_z)}\simeq\frac{H_\mathrm{Data,norm}(v_z)}{H_\mathrm{MC,norm}(v_z)} .$$
The legacy producer comment called $f$ a "data/MC ratio". The code, the coefficients, and the closure below all show that $f$ is MC/Data, and the comment was corrected on 2026-11-27. This notebook keeps the dividing convention.

**Beam-direction correspondence.** In the stored frame (after the flips above), MC Pb-going pairs with data p-going and MC p-going with data Pb-going: in each pair either both samples were flipped or neither was, so the ratio is formed in one detector frame and no $v_z$ reversal is applied. The stored-frame means confirm this (MC Pb-going $-1.40$ cm vs data p-going $-1.01$ cm; MC p-going $+1.41$ cm vs data Pb-going $+1.00$ cm). The CP5 pp sample is Pb-going (see below) and is compared with p-going data.

**CP5 direction (analyst conclusion, 2026-11-25).** The official CP5 sample is treated as **Pb-going**. The CP5 runner passes direction 1 (`isPbGoing`), so its $v_z$ is flipped like the other Pb-going MC. Its stored-frame mean is negative ($-0.57$ cm), as for the Pb-going embedding and PYTHIA samples ($-1.40$ cm). P-going MC has a positive mean ($+1.41$ cm). CP5 is therefore paired with p-going data. Since 2026-11-27, production applies this function to CP5 as well, so the pp MC weight is the stored generator weight times $1/f$.

**Scope.** Weights are extracted only for the direction-specific pairings. Combined samples are not used, and only the post-selection histogram `hVzUnweighted` is analysed. The pre-selection `hVzRaw` includes data events that fail the pPb filters, which pile up near the beam spot.

## Environment and imports

The repository is located dynamically and PyROOT is imported from the active project environment (start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`). ROOT runs in batch mode, and implicit multithreading stays **off** because the event-level split uses ordered entry parity.

In [ ]:
from pathlib import Path
import sys, json, math, copy

PROJECT_ROOT = next(
    (c for c in (Path.cwd(), *Path.cwd().parents)
     if (c / "CMakeLists.txt").is_file() and (c / "hist_analysis").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate the jetAnalysis repository. Start Jupyter from its root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
ROOT.gStyle.SetOptStat(0)
ROOT.gErrorIgnoreLevel = ROOT.kWarning
if ROOT.IsImplicitMTEnabled():
    ROOT.DisableImplicitMT()

from hist_analysis.config.files import BASE_DIR, DATA_DIRS
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_data_file, resolve_direction_file,
    resolve_unfolding_mc_file, open_root_file,
)
from hist_analysis.python.plot_labels import (
    CMS_PRELIMINARY_HEADER, CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT, PPB_MPLHEP_HEADER_RIGHT,
)
from hist_analysis.python.root_style import configure_cms_header, configure_mplhep_export
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection
from hist_analysis.python.notebook_output import NotebookRootOutput
from hist_analysis.python import vz_reweighting as vz
from hist_analysis.python import vz_reweighting_plots as vzp
# Pick up edits to the helper modules in a long-lived kernel (a plain import
# would keep the version loaded when the kernel started).
import importlib
vz = importlib.reload(vz)
vzp = importlib.reload(vzp)

## Configuration

All user-facing parameters are in this cell.

* **Histogram.** `HIST_NAME = 'hVzUnweighted'` is the post-selection distribution of exactly the events that receive the weight in production.
* **Samples.** `MC_SAMPLES` lists the direction-specific MC outputs: embedding and PYTHIA for each beam direction, plus the official CP5 pp sample. Each sample has an `enabled` switch. `MC_TO_DATA_DIRECTION` is the explicit beam-direction mapping. CP5 is Pb-going.
* **MC comparison.** `MC_COMPARISON_SAMPLES` are overlaid, unit-normalized, with their ratios to `MC_COMPARISON_REFERENCE` (Embedding Pb-going).
* **Ranges.** The normalization uses the in-range bins whose centres lie in `NORMALIZATION_RANGE`; under/overflow never enter. The fit uses the valid bins fully inside `FIT_RANGE`. The default $[-15.1,15.1]$ includes all 60 bins. The weight is defined on `WEIGHT_VALID_RANGE`, the $|v_z|\le15$ cm event selection.
* **Styles and ranges.** `PLOT_STYLES` sets the (ROOT colour, ROOT marker) of data and of each MC sample in every figure; reweighted MC uses the open/filled complement of its marker. `RATIO_Y_RANGE` is shared by every MC/Data panel, including the overlay of all fits. `SHOW_FIT_PARAMETERS` prints $p_k\pm\sigma_k$ under the $\chi^2$/NDF line.
* **Plotting.** `USE_MPLHEP` renders native mplhep figures, including the TF1 curves. `SAVE_ROOT_CANVASES` additionally writes the ROOT canvases. The generic ROOT-to-mplhep canvas mirror (`SAVE_MPLHEP`) does not draw TF1s, so it is disabled here to avoid figures without their fits.

In [ ]:
# --- inputs -------------------------------------------------------------
HIST_NAME = 'hVzUnweighted'                # post-selection events, i.e. those that receive the weight
FILE_STEM = 'jetId'                        # production stem; v_z histograms do not depend on the jet selection
DATA_TRIGGER = 'MB'                        # minimum-bias data
PREFER_PRODUCED = True                     # project resolvers: reusable bundles first, then original outputs
VERIFY_AGAINST_ORIGINAL = True             # require bit-identical histograms in bundle and original output

MC_SAMPLES = {
    'embedding_Pbgoing':  dict(generator='embedding', direction='Pbgoing',  enabled=True),
    'embedding_pgoing':   dict(generator='embedding', direction='pgoing',   enabled=True),
    'pythia_Pbgoing':     dict(generator='pythia',    direction='Pbgoing',  enabled=True),
    'pythia_pgoing':      dict(generator='pythia',    direction='pgoing',   enabled=True),
    # Official CP5 pp MC, processed with isPbGoing=1.
    # Pb-going confirmed: negative stored-frame mean like the other Pb-going MC (see introduction).
    'cp5_Pbgoing':        dict(generator='pythia_cp5', direction='Pbgoing', enabled=True),
}
# Explicit MC -> data beam-direction correspondence (stored frame, no v_z reversal).
MC_TO_DATA_DIRECTION = {'Pbgoing': 'pgoing', 'pgoing': 'Pbgoing'}
DATA_DIRECTIONS = tuple(dict.fromkeys(MC_TO_DATA_DIRECTION.values()))

# MC-only shape comparison (unit area), ratios to the reference sample.
MC_COMPARISON_REFERENCE = 'embedding_Pbgoing'
MC_COMPARISON_SAMPLES = ('embedding_Pbgoing', 'embedding_pgoing', 'pythia_Pbgoing', 'pythia_pgoing', 'cp5_Pbgoing')
MC_COMPARISON_RATIO_RANGE = (0.0, 4.5)

# --- ranges and fit -----------------------------------------------------
NORMALIZATION_RANGE = (-15.0, 15.0)        # bin centres; flow bins excluded
FIT_RANGE = (-15.1, 15.1)
WEIGHT_VALID_RANGE = (-15.0, 15.0)         # production |vz| <= 15 cm acceptance
POLY_ORDER = 8
INITIAL_PARAMETERS = vz.LEGACY_POL8_PARAMETERS  # legacy production values as starting values only, never fixed
ORDER_SCAN = (2, 3, 4, 5, 6, 7, 8)
ALTERNATIVE_FUNCTIONS = {                  # diagnostic shapes, not exported to C++
    'exp(pol2)': ('exp([0]+[1]*x+[2]*x*x)', (-0.15, 0.0, 0.008)),
}
CLOSURE_WEIGHT_MODE = 'average'            # 'average' (<1/f> over each bin) or 'centre'
ACCEPTANCE = dict(max_abs_pull=5.0, max_closure_pull=5.0)  # quality flags (in statistical sigma), not physics truth

# --- event-level closure ------------------------------------------------
EVENT_LEVEL_CLOSURE = True
EVENT_LEVEL_SAMPLES = ('embedding_Pbgoing', 'embedding_pgoing', 'pythia_Pbgoing', 'pythia_pgoing', 'cp5_Pbgoing')

# --- plotting and output ------------------------------------------------
USE_MPLHEP = True
SAVE_ROOT_CANVASES = True
SAVE_PNG = False
SAVE_MPLHEP = False                        # generic canvas mirror: no TF1 support, keep off
X_RANGE = (-15.0, 15.0)
DIST_Y_RANGE = (0.0, 0.065)
RATIO_Y_RANGE = (0.0, 10.0)               # every MC/Data ratio panel and the f overlay
# (ROOT colour, ROOT marker) per curve; reweighted MC uses the open/filled complement.
PLOT_STYLES = {
    'data':              (ROOT.kBlack, 20),
    'embedding_Pbgoing': (ROOT.kRed, 21),
    'embedding_pgoing':  (ROOT.kBlue, 22),
    'pythia_Pbgoing':    (ROOT.kOrange + 7, 25),
    'pythia_pgoing':     (ROOT.kMagenta, 26),
    'cp5_Pbgoing':       (ROOT.kPink + 1, 33),
}
FIT_LINE_COLOR = ROOT.kBlack
SHOW_FIT_PARAMETERS = True                 # p_k +- sigma_k under the chi2/NDF line
FIT_TEXT_SCALE = 0.8
WEIGHT_Y_RANGE = (0.0, 2.0)
CLOSURE_RATIO_RANGE = (0.9, 1.1)
PULL_RANGE = (-8.0, 8.0)
CMS_HEADER_LEFT = CMS_PRELIMINARY_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
CMS_HEADER_RIGHT_MPLHEP = PPB_MPLHEP_HEADER_RIGHT
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'vz_reweighting'

HIST_VARIANTS = (HIST_NAME,)               # internal loop variable; one histogram
assert POLY_ORDER >= 0 and FIT_RANGE[0] < FIT_RANGE[1] and NORMALIZATION_RANGE[0] < NORMALIZATION_RANGE[1]
assert set(MC_TO_DATA_DIRECTION) == {'Pbgoing', 'pgoing'}
assert MC_COMPARISON_REFERENCE in MC_COMPARISON_SAMPLES

configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
STYLES = vzp.styles_from_config(PLOT_STYLES)
FIT_STYLE = vzp.CurveStyle(FIT_LINE_COLOR, 20)
configure_mplhep_export(SAVE_MPLHEP, save_png=False)
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OutputSelection(FILE_STEM, None, None, 'ptAve'))
ROOT_OUTPUT = NotebookRootOutput(OUTPUT_DIR, '02_mc_data_vz_reweighting')
print('Histogram variants:', HIST_VARIANTS)
print('Output directory:', OUTPUT_DIR)

## Input files

The files are resolved with the project resolvers. Reusable bundles under `hist_analysis/root_files/produced_histograms/` are preferred and carry provenance metadata; otherwise the original producer outputs are used. For each bundle, the $v_z$ histograms are required to be **bit-identical** to the original producer output, which detects stale bundles.

In [ ]:
def mc_original(sample):
    gen, direction = sample['generator'], sample['direction']
    if gen == 'pythia_cp5':
        return BASE_DIR / 'pythia' / 'cp5tune' / 'HiForestMiniAOD_hist.root'
    return BASE_DIR / gen / direction / f'{gen}_{direction}_{FILE_STEM}.root'

def mc_file(sample):
    gen, direction = sample['generator'], sample['direction']
    if gen == 'pythia_cp5':
        return resolve_unfolding_mc_file(BASE_DIR, 'pythia_cp5', 'pp', FILE_STEM) if PREFER_PRODUCED else mc_original(sample)
    return resolve_direction_file(BASE_DIR, gen, direction, FILE_STEM, prefer_produced=PREFER_PRODUCED)

def data_file(direction, produced=PREFER_PRODUCED):
    return resolve_data_file(DATA_DIRS['combined'], DATA_TRIGGER, direction, FILE_STEM,
                             prefer_produced=produced)

ENABLED_MC = {key: s for key, s in MC_SAMPLES.items() if s['enabled']}
INPUT_FILES = {('mc', key): (mc_file(s), mc_original(s)) for key, s in ENABLED_MC.items()}
for direction in DATA_DIRECTIONS:
    INPUT_FILES[('data', direction)] = (data_file(direction), data_file(direction, produced=False))
for (kind, key), (path, original) in INPUT_FILES.items():
    print(f'{kind:4s} {key:20s} {path}')

## Load and validate the histograms

Every histogram is cloned and detached with `SetDirectory(0)` before it is used. The checks are:

* the object exists, is one-dimensional, has entries, and has no negative bins;
* binning is identical across all samples, and bin widths are uniform;
* each bundle is identical to its original output, including the flow bins.

Uniform 0.5 cm bins mean that normalized bin contents and probability densities differ by a constant factor 2 cm$^{-1}$. Ratios and fits are therefore the same for either, and the export provides both. The table reports the under/overflow fractions, which are excluded from the normalization; they are events at $|v_z|>15$ cm that fail the production selection.

In [ ]:
def load_checked(path, original, name, label):
    histogram = load_histogram(str(path), name)
    histogram.SetName(f'{label}_{name}_input')
    if VERIFY_AGAINST_ORIGINAL and Path(original) != Path(path) and Path(original).is_file():
        reference = load_histogram(str(original), name)
        same = (vz.same_binning(histogram, reference) and all(
            histogram.GetBinContent(i) == reference.GetBinContent(i)
            and histogram.GetBinError(i) == reference.GetBinError(i)
            for i in range(histogram.GetNbinsX() + 2)))
        if not same:
            raise ValueError(f'{label}/{name}: bundle {path} differs from original {original}')
    return histogram

RAW = {}          # RAW[variant][('mc'|'data', key)] -> detached clone of the stored histogram
VALIDATION = {}
for variant in HIST_VARIANTS:
    RAW[variant] = {}
    for (kind, key), (path, original) in INPUT_FILES.items():
        label = f'{kind}_{key}'
        histogram = load_checked(path, original, variant, label)
        RAW[variant][(kind, key)] = histogram
        VALIDATION[(variant, kind, key)] = vz.validate_histogram(histogram, label=f'{variant}:{label}')

reference = next(iter(RAW[HIST_NAME].values()))
for variant in HIST_VARIANTS:
    for (kind, key), histogram in RAW[variant].items():
        if not vz.same_binning(reference, histogram):
            raise ValueError(f'Binning of {variant}/{kind}/{key} differs from the reference')
BINNING = vz.binning_info(reference)
assert BINNING.uniform, 'Non-uniform bins: densities and bin fractions must be distinguished'
print(f'Binning: {BINNING.n_bins} uniform bins of {BINNING.width} cm in [{BINNING.low}, {BINNING.high}] cm')
print(f"{'variant':14s} {'sample':24s} {'entries':>14s} {'in range':>14s} {'flow %':>8s} {'mean':>7s} {'rms':>6s} empty")
for (variant, kind, key), info in VALIDATION.items():
    print(f"{variant:14s} {kind+':'+key:24s} {info['entries']:14.0f} {info['in_range']:14.0f} "
          f"{100*info['flow_fraction']:8.3f} {info['mean']:7.3f} {info['rms']:6.3f} {info['empty_bins']}")

### Is the MC $v_z$ shape independent of the pT-hat sample?

The stored MC histograms sum the pT-hat samples **without** generator weights, but production applies the $v_z$ weight together with the pT-hat weights. This is only equivalent when the $v_z$ shape is the same in every pT-hat sample. The check uses the per-forest histograms from the event-level pass below (stored frame, `isGoodEvent`-like selection) and is printed there.

## Normalization and ratios

For each pairing,
$$p_i=\frac{H_i}{\sum_{j\in\mathrm{norm}}H_j},\qquad R_i=\frac{p_i^\mathrm{MC}}{p_i^\mathrm{Data}},\qquad
\frac{\sigma_{R_i}}{R_i}=\sqrt{\Big(\frac{\sigma_{p_i^\mathrm{MC}}}{p_i^\mathrm{MC}}\Big)^2+\Big(\frac{\sigma_{p_i^\mathrm{Data}}}{p_i^\mathrm{Data}}\Big)^2},$$
and the Data/MC ratio $W_i=1/R_i$ has the same relative error. MC and data are independent samples. Normalization rescales the Poisson errors by the fixed integral. The multinomial correlation introduced by the shared denominator is neglected; its relative size is $\mathcal{O}(N^{-1/2})$, below $5\times10^{-4}$ for the $\ge4\times10^{6}$ entries used here.

A bin is valid only if both contents and both errors are positive and finite. Invalid bins are set to zero, listed, and excluded from fits and closure metrics. Under/overflow never enter.

In [ ]:
NORM_BINS = vz.bin_range(reference, NORMALIZATION_RANGE)

def pairing_definitions():
    pairs = {}
    for key, sample in ENABLED_MC.items():
        data_direction = MC_TO_DATA_DIRECTION[sample['direction']]
        pairs[key] = dict(mc=key, data=data_direction, mc_direction=sample['direction'],
                          generator=sample['generator'],
                          diagnostic_only=sample.get('diagnostic_only', False),
                          assumed_direction=sample.get('assumed_direction', False))
    return pairs

PAIRS = pairing_definitions()
NORMALIZED = {}   # NORMALIZED[variant][(kind, key)]
RATIOS = {}       # RATIOS[variant][pair] -> dict
for variant in HIST_VARIANTS:
    normalized = {}
    for (kind, key), histogram in RAW[variant].items():
        normalized[(kind, key)], _ = vz.normalize_unit_area(
            histogram, f'h_{variant}_{kind}_{key}_norm', norm_range=NORMALIZATION_RANGE)
    NORMALIZED[variant] = normalized
    RATIOS[variant] = {}
    for pair_key, pair in PAIRS.items():
        mc, data = normalized[('mc', pair['mc'])], normalized[('data', pair['data'])]
        mc_over_data, invalid = vz.ratio_histogram(mc, data, f'h_{variant}_{pair_key}_mcOverData', bins=NORM_BINS)
        data_over_mc, invalid2 = vz.ratio_histogram(data, mc, f'h_{variant}_{pair_key}_dataOverMc', bins=NORM_BINS)
        RATIOS[variant][pair_key] = dict(mc_over_data=mc_over_data, data_over_mc=data_over_mc,
                                         invalid_bins=sorted(set(invalid) | set(invalid2)))
for pair_key, pair in PAIRS.items():
    print(f"  {pair_key:26s} MC {pair['mc_direction']:8s} -> data {pair['data']:8s}"
          f"{'  [assumed direction]' if pair['assumed_direction'] else ''}"
          f"{'  [diagnostic only]' if pair['diagnostic_only'] else ''}"
          f"  invalid bins: {RATIOS[HIST_NAME][pair_key]['invalid_bins']}")

## Polynomial fits of the MC/Data ratio

For each variant and pairing, the MC/Data ratio is fitted with `pol<POLY_ORDER>` over `FIT_RANGE` with a $\chi^2$ fit, using only the valid bins at their bin centres. The fit is done on a `TGraphErrors` with zero $x$ errors (`EX0`). Non-zero $x$ errors would inflate the effective variance where the ratio is steep and change the bin weights.

* ROOT fits `polN` with its linear fitter unless option `F` is given. `F` (MINUIT) is used here so that `INITIAL_PARAMETERS` act as genuine starting values; nothing is fixed. The problem is linear in the parameters, so the starting values should not change the minimum.
* That expectation is checked: an exact weighted least-squares solution, computed in the scaled variable $u=v_z/15$ to avoid the ill-conditioned raw powers, is compared with the MINUIT result.
* For the scan orders, the least-squares solution is the starting point.

Everything is recorded: fit and covariance status, parameters, errors, covariance, $\chi^2$, NDF, and probability.

When $\chi^2/\mathrm{NDF}\gg1$, the parameter errors are not meaningful at face value. They are also shown scaled by $\sqrt{\chi^2/\mathrm{NDF}}$, and the fit is flagged.

In [ ]:
FITS = {}           # FITS[variant][pair] -> vz.FitResult (default order)
SCANS = {}          # SCANS[variant][pair][order] -> FitResult
ALTERNATIVES = {}
STABILITY = {}
for variant in HIST_VARIANTS:
    FITS[variant], SCANS[variant], ALTERNATIVES[variant], STABILITY[variant] = {}, {}, {}, {}
    for pair_key in PAIRS:
        ratio = RATIOS[variant][pair_key]['mc_over_data']
        tag = f'{variant}_{pair_key}'
        FITS[variant][pair_key] = vz.fit_polynomial(
            ratio, name=f'fVzWeight_{tag}', order=POLY_ORDER, fit_range=FIT_RANGE,
            start=INITIAL_PARAMETERS if POLY_ORDER == 8 else None)
        SCANS[variant][pair_key] = {
            order: vz.fit_polynomial(ratio, name=f'fVzScan_pol{order}_{tag}', order=order,
                                     fit_range=FIT_RANGE,
                                     start=INITIAL_PARAMETERS if order == 8 else None)
            for order in ORDER_SCAN}
        ALTERNATIVES[variant][pair_key] = {
            label: vz.fit_function(ratio, name=f'fVzAlt_{i}_{tag}', formula=formula,
                                   fit_range=FIT_RANGE, start=start)
            for i, (label, (formula, start)) in enumerate(ALTERNATIVE_FUNCTIONS.items())}
        STABILITY[variant][pair_key] = vz.stability_checks(
            FITS[variant][pair_key].function, check_range=WEIGHT_VALID_RANGE,
            ratio=ratio, fit_range=FIT_RANGE)

def fit_table(variant):
    print(f'\n=== {variant}: pol{POLY_ORDER} fits of MC/Data ===')
    print(f"{'pairing':26s} {'status':>6s} {'cov':>4s} {'chi2':>10s} {'ndf':>4s} {'chi2/ndf':>9s} "
          f"{'max|pull|':>9s} {'min f':>7s} {'f>0':>4s} {'extrema':>7s} {'MINUIT-LSQ':>10s}")
    for pair_key, fit in FITS[variant].items():
        s = STABILITY[variant][pair_key]
        print(f"{pair_key:26s} {fit.status:6d} {fit.cov_status:4d} {fit.chi2:10.1f} {fit.ndf:4d} "
              f"{fit.chi2_ndf:9.2f} {s['max_abs_pull']:9.2f} {s['min_value']:7.4f} "
              f"{str(s['all_positive']):>4s} {s['extremum_count']:7d} {fit.max_linear_relative_difference:10.1e}")

for variant in HIST_VARIANTS:
    fit_table(variant)

In [ ]:
for variant in HIST_VARIANTS:
    print(f'\n=== {variant}: fitted parameters (errors; errors x sqrt(chi2/ndf) when chi2/ndf > 1) ===')
    for pair_key, fit in FITS[variant].items():
        scale = math.sqrt(fit.chi2_ndf) if fit.chi2_ndf > 1 else 1.0
        print(f'{pair_key}:')
        for k, (p, e) in enumerate(zip(fit.parameters, fit.errors)):
            print(f'   p{k} = {p:+.6e} +- {e:.2e}' + (f'  (scaled {e*scale:.2e})' if scale > 1 else ''))

### Polynomial-order scan and alternative shape

For each order, the scan reports $\chi^2/\mathrm{NDF}$, the maximum $|$pull$|$, and the number of local extrema of $f$ in $[-15,15]$ cm; more than one extremum indicates oscillation. It also reports the nested-model $F$-test $p$-value for adding the extra parameter(s) relative to the previous order. The $F$-test assumes the larger model describes the data, so it is only indicative where $\chi^2/\mathrm{NDF}\gg1$.

`exp(pol2)` is the exact form of a ratio of two Gaussians. It is shown as a diagnostic of whether the shapes are Gaussian; it is not exported.

In [ ]:
ORDER_SCAN_SUMMARY = {}
for variant in HIST_VARIANTS:
    print(f'\n=== {variant}: order scan ===')
    ORDER_SCAN_SUMMARY[variant] = {}
    for pair_key, scan in SCANS[variant].items():
        rows, previous = [], None
        for order, fit in scan.items():
            s = vz.stability_checks(fit.function, check_range=WEIGHT_VALID_RANGE,
                                    ratio=RATIOS[variant][pair_key]['mc_over_data'], fit_range=FIT_RANGE)
            test = vz.f_test(previous.chi2, previous.ndf, fit.chi2, fit.ndf) if previous else dict(p_value=float('nan'))
            rows.append(dict(order=order, chi2=fit.chi2, ndf=fit.ndf, chi2_ndf=fit.chi2_ndf,
                             max_abs_pull=s['max_abs_pull'], extrema=s['extremum_count'],
                             positive=s['all_positive'], f_test_p=test['p_value'], status=fit.status))
            previous = fit
        for label, fit in ALTERNATIVES[variant][pair_key].items():
            rows.append(dict(order=label, chi2=fit.chi2, ndf=fit.ndf, chi2_ndf=fit.chi2_ndf,
                             max_abs_pull=float('nan'), extrema=-1, positive=True,
                             f_test_p=float('nan'), status=fit.status))
        ORDER_SCAN_SUMMARY[variant][pair_key] = rows
        print(pair_key)
        for r in rows:
            print(f"   {str(r['order']):>9s}  chi2/ndf = {r['chi2']:10.1f}/{r['ndf']:2d} = {r['chi2_ndf']:8.2f}"
                  f"  max|pull| = {r['max_abs_pull']:6.2f}  extrema = {r['extrema']:2d}"
                  f"  F-test p = {r['f_test_p']:.2e}  status = {r['status']}")

## Multiplicative weight and histogram-level closure

The weight is $w(v_z)=1/f(v_z)$, which is exactly what `eventWeight` applies. It is defined on `WEIGHT_VALID_RANGE`, the production $|v_z|\le15$ cm acceptance, and the exported TF1 has that range. **Outside this range the function must not be evaluated.** Production cannot evaluate it there, because such events fail `isGoodEvent`. Any other use should reject those events rather than extrapolate the polynomial.

At histogram level, each MC bin is multiplied by $\langle 1/f\rangle$ averaged over the bin (`CLOSURE_WEIGHT_MODE='average'`), renormalized, and compared with data. Agreement is quantified before and after reweighting by these metrics, all over the normalization bins:

* $\chi^2=\sum_i (p^\mathrm{MC}_i-p^\mathrm{D}_i)^2/(\sigma_\mathrm{MC}^2+\sigma_\mathrm{D}^2)$;
* $\sum|p^\mathrm{MC}-p^\mathrm{D}|$;
* $\max_i|p^\mathrm{MC}_i/p^\mathrm{D}_i-1|$.

**Interpretation.** $f$ was fitted to these same two histograms, so this closure is circular: it is a goodness-of-fit statement, and the residual after reweighting is the fit residual. The independent test is the event-level closure in the next section. It also matters that the bin average of $1/f$ is uniform within each bin, while events follow the steep MC density; the event-level pass quantifies that approximation.

In [ ]:
WEIGHTS, REWEIGHTED, CLOSURE = {}, {}, {}
for variant in HIST_VARIANTS:
    WEIGHTS[variant], REWEIGHTED[variant], CLOSURE[variant] = {}, {}, {}
    for pair_key, pair in PAIRS.items():
        fit = FITS[variant][pair_key]
        WEIGHTS[variant][pair_key] = vz.weight_function(
            fit.function, f'fMcVzWeight_{variant}_{pair_key}', valid_range=WEIGHT_VALID_RANGE)
        mc = NORMALIZED[variant][('mc', pair['mc'])]
        data = NORMALIZED[variant][('data', pair['data'])]
        reweighted = vz.reweight_histogram(mc, fit.function, f'h_{variant}_{pair_key}_mcReweighted',
                                           bins=NORM_BINS, mode=CLOSURE_WEIGHT_MODE)
        reweighted_norm, _ = vz.normalize_unit_area(reweighted, f'h_{variant}_{pair_key}_mcReweightedNorm',
                                                    norm_range=NORMALIZATION_RANGE)
        closure_ratio, _ = vz.ratio_histogram(reweighted_norm, data, f'h_{variant}_{pair_key}_closureRatio',
                                              bins=NORM_BINS)
        REWEIGHTED[variant][pair_key] = dict(norm=reweighted_norm, closure_ratio=closure_ratio)
        before = vz.agreement(mc, data, bins=NORM_BINS)
        after = vz.agreement(reweighted_norm, data, bins=NORM_BINS)
        CLOSURE[variant][pair_key] = dict(before=before, after=after)

    print(f'\n=== {variant}: histogram-level closure (circular; see text) ===')
    print(f"{'pairing':26s} {'chi2/ndf before':>16s} {'after':>10s} {'sum|d| before':>14s} {'after':>8s} "
          f"{'max|r-1| before':>16s} {'after':>8s} {'max|pull| after':>15s} {'mean MC->rw (data)':>22s}")
    for pair_key, c in CLOSURE[variant].items():
        b, a = c['before'], c['after']
        print(f"{pair_key:26s} {b['chi2_ndf']:16.1f} {a['chi2_ndf']:10.2f} {b['sum_abs_difference']:14.4f} "
              f"{a['sum_abs_difference']:8.4f} {b['max_abs_ratio_minus_one']:16.3f} {a['max_abs_ratio_minus_one']:8.4f} {a['max_abs_pull']:15.2f} "
              f"{b['mean_mc']:+7.3f}->{a['mean_mc']:+7.3f} ({a['mean_data']:+.3f})")

### Legacy production function

The same closure is repeated with the single legacy function that `processForestSimple.C` applied to every pPb MC sample until 2026-11-27, using the production convention $1/f$. It shows how much the generator- and direction-specific functions improve on it. Before that date, CP5 (pp) had no $v_z$ weight at all, so its "none" column is what the old production used.

In [ ]:
PRODUCTION_FUNCTION = ROOT.TF1('fVzWeight_production_legacy', 'pol8', -15.1, 15.1)
for k, value in enumerate(vz.LEGACY_POL8_PARAMETERS):
    PRODUCTION_FUNCTION.SetParameter(k, value)
PRODUCTION_CLOSURE = {}
for variant in HIST_VARIANTS:
    PRODUCTION_CLOSURE[variant] = {}
    print(f'\n=== {variant}: closure with the legacy single fVzWeight (divide) ===')
    for pair_key, pair in PAIRS.items():
        mc = NORMALIZED[variant][('mc', pair['mc'])]
        data = NORMALIZED[variant][('data', pair['data'])]
        rw = vz.reweight_histogram(mc, PRODUCTION_FUNCTION, f'h_{variant}_{pair_key}_prodRw', bins=NORM_BINS)
        rw, _ = vz.normalize_unit_area(rw, f'h_{variant}_{pair_key}_prodRwNorm', norm_range=NORMALIZATION_RANGE)
        PRODUCTION_CLOSURE[variant][pair_key] = vz.agreement(rw, data, bins=NORM_BINS)
        b, a = CLOSURE[variant][pair_key]['before'], PRODUCTION_CLOSURE[variant][pair_key]
        grid = [WEIGHT_VALID_RANGE[0] + 0.05 * k for k in range(601)]
        a['max_rel_diff_to_new_fit'] = max(abs(PRODUCTION_FUNCTION.Eval(x) / FITS[variant][pair_key].function.Eval(x) - 1) for x in grid)
        print(f"{pair_key:26s} |f_prod/f_new - 1| <= {a['max_rel_diff_to_new_fit']:.3f};"
              f" sum|d|: none {b['sum_abs_difference']:.4f} -> production {a['sum_abs_difference']:.4f}"
              f" -> new fit {CLOSURE[variant][pair_key]['after']['sum_abs_difference']:.4f};"
              f" max|r-1| production {a['max_abs_ratio_minus_one']:.3f}")

## Event-level closure (independent halves)

This test reads `hiEvtAnalyzer/HiTree/vz` directly from the MC HiForest files with RDataFrame, without implicit MT, and applies the producer's flip ($v_z\to-v_z$ for Pb-going MC).

1. **Provenance.** The `isGoodEvent` selection is reproduced: $|v_z|\le15$ cm, the pPb skim filters with the nominal `pVertexFilterCutdz1p0`, and the file's pT-hat range $[\mathrm{low},\mathrm{high})$. The overweight-event rejection needs jets and is not reproduced, so the event-level selected sample is a small superset. The excess fraction is reported.
2. **Independent closure.** Each file is split by entry parity. $f$ is fitted on the even half against data. Each odd-half event is weighted by $1/f_\mathrm{even}(v_z)$, and the result is compared with data. The MC halves are statistically independent. The data distribution is shared, but its statistical error is negligible ($\ge2\times10^9$ entries). Comparing the odd-half event-level result with the histogram-level reweighting of the same odd half measures the bin-averaging approximation.
3. **pT-hat independence.** The per-forest $v_z$ means and RMS values of the selected events test whether the $v_z$ shape depends on the pT-hat sample.

In [ ]:
def forest_files(sample):
    if sample['generator'] == 'pythia_cp5':
        folder = BASE_DIR / 'pythia' / 'cp5tune' / 'forest'
    else:
        folder = BASE_DIR / sample['generator'] / sample['direction'] / 'forest'
    return sorted(folder.glob('*.root'))

EVENT_LEVEL = {}
PTHAT_DEPENDENCE = {}
if EVENT_LEVEL_CLOSURE:
    edges = [reference.GetXaxis().GetBinLowEdge(i) for i in range(1, reference.GetNbinsX() + 2)]
    for variant in HIST_VARIANTS:
        selection = {'hVzUnweighted': 'selected'}.get(variant)
        if selection is None:
            print(f'{variant}: no event-level definition; skipped'); continue
        EVENT_LEVEL[variant] = {}
        for key in EVENT_LEVEL_SAMPLES:
            if key not in ENABLED_MC:
                continue
            sample = ENABLED_MC[key]
            files = forest_files(sample)
            if not files:
                print(f'{variant}/{key}: no forests found; skipped'); continue
            flip = sample['direction'] == 'Pbgoing'
            pp = sample['generator'] == 'pythia_cp5'  # pp selection: |vz| <= 15 cm only
            name = f'ev_{variant}_{key}'
            first = vz.event_level_histograms(files, flip=flip, name=name, edges=edges, selection=selection, pp=pp)
            stored = RAW[variant][('mc', key)]
            all_events = first['all']
            max_diff = max(abs(all_events.GetBinContent(i) - stored.GetBinContent(i))
                           for i in range(all_events.GetNbinsX() + 2))
            excess = all_events.Integral(*NORM_BINS) / stored.Integral(*NORM_BINS) - 1.0
            # Fit on the even half
            data = NORMALIZED[variant][('data', MC_TO_DATA_DIRECTION[sample['direction']])]
            even, _ = vz.normalize_unit_area(first['even'], f'{name}_evenNorm', norm_range=NORMALIZATION_RANGE)
            even_ratio, _ = vz.ratio_histogram(even, data, f'{name}_evenRatio', bins=NORM_BINS)
            even_fit = vz.fit_polynomial(even_ratio, name=f'{name}_evenFit', order=POLY_ORDER,
                                         fit_range=FIT_RANGE,
                                         start=INITIAL_PARAMETERS if POLY_ORDER == 8 else None)
            second = vz.event_level_histograms(files, flip=flip, name=f'{name}_w', edges=edges,
                                               selection=selection, weight_function=even_fit.function, pp=pp)
            odd, _ = vz.normalize_unit_area(second['odd'], f'{name}_oddNorm', norm_range=NORMALIZATION_RANGE)
            odd_event_rw, _ = vz.normalize_unit_area(second['odd_weighted'], f'{name}_oddEventRw',
                                                     norm_range=NORMALIZATION_RANGE)
            odd_hist_rw = vz.reweight_histogram(odd, even_fit.function, f'{name}_oddHistRw', bins=NORM_BINS)
            odd_hist_rw, _ = vz.normalize_unit_area(odd_hist_rw, f'{name}_oddHistRwNorm', norm_range=NORMALIZATION_RANGE)
            binning_effect = max(abs(odd_hist_rw.GetBinContent(i) / odd_event_rw.GetBinContent(i) - 1)
                                 for i in range(NORM_BINS[0], NORM_BINS[1] + 1) if odd_event_rw.GetBinContent(i) > 0)
            closure_ratio, _ = vz.ratio_histogram(odd_event_rw, data, f'{name}_oddClosureRatio', bins=NORM_BINS)
            EVENT_LEVEL[variant][key] = dict(
                n_files=len(files), n_tree_entries=first['n_tree_entries'],
                stored_entries=float(stored.GetEntries()),
                provenance_max_bin_difference=float(max_diff), selected_excess_fraction=float(excess),
                even_fit=even_fit, before=vz.agreement(odd, data, bins=NORM_BINS),
                after=vz.agreement(odd_event_rw, data, bins=NORM_BINS),
                hist_level_vs_event_level_max_rel=float(binning_effect),
                odd_norm=odd, odd_event_rw=odd_event_rw, closure_ratio=closure_ratio)
            if variant == HIST_NAME and not pp:
                PTHAT_DEPENDENCE[key] = {fname: dict(mean=h.GetMean(), rms=h.GetRMS(), entries=h.GetEntries())
                                         for fname, h in first['per_file'].items()}
            r = EVENT_LEVEL[variant][key]
            print(f"{variant:14s} {key:18s} files {r['n_files']:2d} tree entries {r['n_tree_entries']} "
                  f"(stored {r['stored_entries']:.0f}); max|bin diff| {r['provenance_max_bin_difference']:.0f}; "
                  f"in-range excess {100*r['selected_excess_fraction']:+.2f}%")
            print(f"{'':33s} odd half chi2/ndf {r['before']['chi2_ndf']:.1f} -> {r['after']['chi2_ndf']:.2f};"
                  f" max|r-1| {r['before']['max_abs_ratio_minus_one']:.3f} -> {r['after']['max_abs_ratio_minus_one']:.4f};"
                  f" hist-vs-event reweighting max rel. diff {r['hist_level_vs_event_level_max_rel']:.1e}")
    print('\npT-hat dependence of the MC v_z shape (selected events, stored frame):')
    for key, rows in PTHAT_DEPENDENCE.items():
        means = [r['mean'] for r in rows.values()]
        rmss = [r['rms'] for r in rows.values()]
        print(f'  {key:18s} mean {min(means):+.3f} .. {max(means):+.3f} cm; rms {min(rmss):.3f} .. {max(rmss):.3f} cm '
              f'over {len(rows)} pT-hat samples')
        for fname, r in rows.items():
            print(f"      {fname:60s} N={r['entries']:9.0f} mean {r['mean']:+.4f} rms {r['rms']:.4f}")

## Figures

For each histogram variant and pairing, the notebook draws:

1. normalized distributions with an MC/Data panel;
2. the MC/Data ratio with the fit, the unity line, and the fit quality, plus a pull panel;
3. the multiplicative weight $1/f$ compared with the binned Data/MC;
4. the histogram-level closure.

It also draws overlays of all fitted $f$ and of all weights. A sample has the same colour and marker in every figure. Data is black, and reweighted MC uses the open version of its sample's marker. ROOT figures go to `.../root/`, and native mplhep figures to the sibling `.../mplhep/` directory.

In [ ]:
def label_of(pair_key, pair):
    names = {'embedding': 'Embedding', 'pythia': 'PYTHIA', 'pythia_cp5': 'CP5 (pp)'}
    gen = names[pair['generator']]
    direction = {'Pbgoing': 'Pb-going', 'pgoing': 'p-going'}[pair['mc_direction']]
    return f'{gen}, {direction}' + (' (assumed)' if pair['assumed_direction'] else '')

def data_label(direction):
    return 'Data MB, ' + {'Pbgoing': 'Pb-going', 'pgoing': 'p-going'}[direction]

def render(figure, cms_left=None):
    outputs = {}
    left = cms_left or CMS_HEADER_LEFT
    if SAVE_ROOT_CANVASES:
        outputs['root'] = vzp.draw_root(figure, OUTPUT_DIR / f'{figure.name}.pdf',
                                        cms_left=left, cms_right=CMS_HEADER_RIGHT, save_png=SAVE_PNG)
    if USE_MPLHEP:
        import matplotlib.pyplot as plt
        fig = vzp.draw_mplhep(figure, OUTPUT_DIR / 'mplhep' / f'{figure.name}.pdf',
                              cms_left=left, cms_right=CMS_HEADER_RIGHT_MPLHEP, save_png=SAVE_PNG)
        plt.close(fig)  # saved; the object stays displayable
        outputs['mplhep'] = fig
    return outputs

X_TITLE = 'v_{z} (cm)'
FIGURES = {}
for variant in HIST_VARIANTS:
    for pair_key, pair in PAIRS.items():
        style = STYLES[pair_key]
        mc = NORMALIZED[variant][('mc', pair['mc'])]
        data = NORMALIZED[variant][('data', pair['data'])]
        ratio = RATIOS[variant][pair_key]['mc_over_data']
        fit = FITS[variant][pair_key]
        stab = STABILITY[variant][pair_key]
        tag = f'{variant}_{pair_key}'
        pretty = {'Pbgoing': 'Pb-going', 'pgoing': 'p-going'}
        context = [variant, f'MC {pretty[pair["mc_direction"]]} vs data {pretty[pair["data"]]}']
        fit_text = [f'pol{POLY_ORDER}: #chi^{{2}}/NDF = {fit.chi2:.1f}/{fit.ndf}']
        if SHOW_FIT_PARAMETERS:
            fit_text += [f'p_{{{k}}} = {p:.4e} #pm {e:.1e}' for k, (p, e) in enumerate(zip(fit.parameters, fit.errors))]
        fit_text.append(f'max |pull| = {stab["max_abs_pull"]:.1f}')
        pulls = vz.pull_histogram(ratio, fit.function, f'h_{tag}_pulls', fit_range=FIT_RANGE)
        figures = {
            'distributions': vzp.Figure([
                vzp.Panel([vzp.Item('hist', data, data_label(pair['data']), STYLES['data']),
                           vzp.Item('hist', mc, label_of(pair_key, pair), style)],
                          'Normalized counts', DIST_Y_RANGE, text=context),
                vzp.Panel([vzp.Item('hist', ratio, None, style), vzp.Item('hline', value=1.0)],
                          'MC/Data', RATIO_Y_RANGE, legend=False)],
                X_TITLE, X_RANGE, f'vz_distributions_{tag}'),
            'ratio_fit': vzp.Figure([
                vzp.Panel([vzp.Item('hist', ratio, 'MC/Data', style),
                           vzp.Item('func', fit.function, f'pol{POLY_ORDER} fit', FIT_STYLE),
                           vzp.Item('hline', value=1.0)],
                          'MC/Data', RATIO_Y_RANGE, text=[label_of(pair_key, pair), *fit_text],
                          text_scale=FIT_TEXT_SCALE),
                vzp.Panel([vzp.Item('hist', pulls, None, style), vzp.Item('hline', value=0.0)],
                          'Pull', PULL_RANGE, legend=False)],
                X_TITLE, X_RANGE, f'vz_ratio_fit_{tag}'),
            'weight': vzp.Figure([
                vzp.Panel([vzp.Item('hist', RATIOS[variant][pair_key]['data_over_mc'], 'Data/MC (binned)', style),
                           vzp.Item('func', WEIGHTS[variant][pair_key], 'w_{MC} = 1/f', FIT_STYLE)],
                          'MC event weight w_{MC}', WEIGHT_Y_RANGE,
                          text=[label_of(pair_key, pair), variant], legend_bounds=(0.60, 0.70, 0.93, 0.88))],
                X_TITLE, X_RANGE, f'vz_weight_{tag}'),
            'closure': vzp.Figure([
                vzp.Panel([vzp.Item('hist', data, data_label(pair['data']), STYLES['data']),
                           vzp.Item('hist', mc, label_of(pair_key, pair), style),
                           vzp.Item('hist', REWEIGHTED[variant][pair_key]['norm'], 'reweighted MC', vzp.open_style(style))],
                          'Normalized counts', DIST_Y_RANGE, text=context),
                vzp.Panel([vzp.Item('hist', REWEIGHTED[variant][pair_key]['closure_ratio'], None, vzp.open_style(style)),
                           vzp.Item('hline', value=1.0)],
                          'Rew. MC/Data', CLOSURE_RATIO_RANGE, legend=False)],
                X_TITLE, X_RANGE, f'vz_closure_{tag}'),
        }
        FIGURES[(variant, pair_key)] = {name: render(f) for name, f in figures.items()}
        FIGURES[(variant, pair_key)]['_pulls'] = pulls

    # Combined comparison of every pairing's fitted ratio and weight.
    first_ratio = next(iter(RATIOS[variant].values()))['mc_over_data']
    frame = first_ratio.Clone(f'h_{variant}_frame'); frame.Reset('ICES')
    for name, objects, y_title, y_range in (
            ('fits', {k: FITS[variant][k].function for k in PAIRS}, 'f = MC/Data fit', RATIO_Y_RANGE),
            ('weights', WEIGHTS[variant], 'w_{MC} = 1/f', WEIGHT_Y_RANGE)):
        items = [vzp.Item('hist', frame, None, STYLES['data'])]
        items += [vzp.Item('func', obj, label_of(k, PAIRS[k]), STYLES[k],
                           line_style=2 if PAIRS[k]['diagnostic_only'] else 1) for k, obj in objects.items()]
        items.append(vzp.Item('hline', value=1.0))
        top = y_range[1] if name == 'fits' else y_range[1] * 1.4  # MC/Data panels share RATIO_Y_RANGE
        FIGURES[(variant, 'overlay_' + name)] = render(vzp.Figure(
            [vzp.Panel(items, y_title, (y_range[0], top), legend_bounds=(0.22, 0.62, 0.95, 0.88),
                       legend_columns=1, legend_header=variant)],
            X_TITLE, X_RANGE, f'vz_all_{name}_{variant}'))

import matplotlib.pyplot as plt
from IPython.display import display
for key in [(HIST_NAME, 'embedding_Pbgoing'), (HIST_NAME, 'embedding_pgoing')]:
    if key in FIGURES and 'mplhep' in FIGURES[key]['ratio_fit']:
        display(FIGURES[key]['distributions']['mplhep']); display(FIGURES[key]['ratio_fit']['mplhep'])
        display(FIGURES[key]['weight']['mplhep']); display(FIGURES[key]['closure']['mplhep'])
for variant in HIST_VARIANTS:
    if 'mplhep' in FIGURES[(variant, 'overlay_weights')]:
        display(FIGURES[(variant, 'overlay_fits')]['mplhep']); display(FIGURES[(variant, 'overlay_weights')]['mplhep'])
plt.close('all')

## MC sample comparison

This section overlays the unit-normalized `hVzUnweighted` distributions of the MC samples and divides each by the reference, Embedding Pb-going. The official CP5 pp sample is included. All shapes are in the stored frame, exactly as produced, so p-going MC appears mirrored relative to Pb-going MC (means of about $\pm1.4$ cm). Its ratio to Embedding Pb-going reflects that flip and should not be read as a generator difference. The samples are statistically independent, so ratio errors use independent propagation. The $\chi^2$ against the reference counts both samples' errors, over all in-range bins.

In [ ]:
MC_COMPARISON = {}
reference_mc = NORMALIZED[HIST_NAME][('mc', MC_COMPARISON_REFERENCE)]
top_items, ratio_items = [], []
for key in MC_COMPARISON_SAMPLES:
    if ('mc', key) not in NORMALIZED[HIST_NAME]:
        print(f'{key}: not enabled; skipped'); continue
    histogram = NORMALIZED[HIST_NAME][('mc', key)]
    style = STYLES[key]
    top_items.append(vzp.Item('hist', histogram, label_of(key, PAIRS[key]), style))
    entry = dict(mean=histogram.GetMean(), rms=histogram.GetRMS(),
                 entries=float(RAW[HIST_NAME][('mc', key)].GetEntries()))
    if key != MC_COMPARISON_REFERENCE:
        ratio, invalid = vz.ratio_histogram(histogram, reference_mc,
                                            f'h_{HIST_NAME}_{key}_over_{MC_COMPARISON_REFERENCE}', bins=NORM_BINS)
        ratio_items.append(vzp.Item('hist', ratio, None, style))
        entry.update(ratio=ratio, invalid_bins=invalid,
                     agreement=vz.agreement(histogram, reference_mc, bins=NORM_BINS))
    MC_COMPARISON[key] = entry
ratio_items.append(vzp.Item('hline', value=1.0))
reference_label = label_of(MC_COMPARISON_REFERENCE, PAIRS[MC_COMPARISON_REFERENCE])
FIGURES[(HIST_NAME, 'mc_comparison')] = render(vzp.Figure(
    [vzp.Panel(top_items, 'Normalized counts', DIST_Y_RANGE,
               text=[HIST_NAME, 'MC only, stored frame', 'ref. = Emb. Pb-going']),
     vzp.Panel(ratio_items, 'MC / ref.', MC_COMPARISON_RATIO_RANGE, legend=False)],
    X_TITLE, X_RANGE, f'vz_mc_comparison_{HIST_NAME}'), cms_left=CMS_SIMULATION_HEADER)

print(f'Reference: {reference_label}')
print(f"{'sample':20s} {'entries':>10s} {'mean':>8s} {'rms':>7s} {'chi2/ndf vs ref':>16s} {'max|r-1|':>9s}")
for key, entry in MC_COMPARISON.items():
    a = entry.get('agreement')
    print(f"{key:20s} {entry['entries']:10.0f} {entry['mean']:+8.3f} {entry['rms']:7.3f} "
          + (f"{a['chi2_ndf']:16.2f} {a['max_abs_ratio_minus_one']:9.3f}" if a else f"{'reference':>16s}"))
if 'mplhep' in FIGURES[(HIST_NAME, 'mc_comparison')]:
    display(FIGURES[(HIST_NAME, 'mc_comparison')]['mplhep'])

## Export

The results file `vz_reweighting_results.root` contains one directory per histogram variant and pairing, holding:

* the original histograms (`hMcInput`, `hDataInput`);
* the normalized histograms (`hMcNorm`, `hDataNorm`) and densities (`hMcDensity`, `hDataDensity`);
* the ratios `hMcOverData` and `hDataOverMc`;
* the fitted MC/Data TF1 `fVzWeight` and the multiplicative weight TF1 `fMcWeight` $=1/f$, defined on $|v_z|\le15$ cm only;
* the order-scan TF1s;
* the reweighted MC (`hMcReweightedNorm`), the closure ratio, and the pulls;
* `fitSummary`, a JSON `TNamed` with the parameters, errors, covariance, $\chi^2$, NDF, ranges, mapping and closure metrics.

Alongside it are written `vz_reweighting_summary.json`, a human-readable `vz_reweighting_summary.md`, and `vz_reweighting_fVzWeight.C` with drop-in C++ snippets. The snippets state explicitly that the MC event weight **divides** by `fVzWeight`.

In [ ]:
def fit_record(variant, pair_key):
    pair, fit = PAIRS[pair_key], FITS[variant][pair_key]
    stab = {k: v for k, v in STABILITY[variant][pair_key].items() if k != 'pulls'}
    return dict(
        histogram=variant, pairing=pair_key, mc_sample=pair['mc'], mc_direction=pair['mc_direction'],
        data_direction=pair['data'], data_trigger=DATA_TRIGGER, assumed_direction=pair['assumed_direction'],
        diagnostic_only=pair['diagnostic_only'],
        normalization_range=list(NORMALIZATION_RANGE), normalization_bins=list(NORM_BINS),
        fit_range=list(FIT_RANGE), weight_valid_range=list(WEIGHT_VALID_RANGE),
        initial_parameters=list(INITIAL_PARAMETERS) if POLY_ORDER == 8 else fit.linear_parameters,
        fit=fit.summary(), stability=stab, invalid_bins=RATIOS[variant][pair_key]['invalid_bins'],
        closure=CLOSURE[variant][pair_key], legacy_function_closure=PRODUCTION_CLOSURE[variant][pair_key],
        order_scan=ORDER_SCAN_SUMMARY[variant][pair_key],
        convention='fVzWeight = fitted MC/Data; MC event weight = 1/fVzWeight(vz) (divide), as eventWeight()',
        inputs=dict(mc=str(INPUT_FILES[('mc', pair['mc'])][0]),
                    data=str(INPUT_FILES[('data', pair['data'])][0])),
    )

SUMMARY = {f'{v}/{p}': fit_record(v, p) for v in HIST_VARIANTS for p in PAIRS}
if EVENT_LEVEL_CLOSURE:
    for variant, rows in EVENT_LEVEL.items():
        for key, r in rows.items():
            SUMMARY[f'{variant}/{key}']['event_level_closure'] = {
                k: (v.summary() if k == 'even_fit' else v) for k, v in r.items()
                if not hasattr(v, 'InheritsFrom')}
    SUMMARY['_pthat_dependence'] = PTHAT_DEPENDENCE

RESULTS_FILE = OUTPUT_DIR / 'vz_reweighting_results.root'
with ROOT.TDirectory.TContext():
    out = open_root_file(str(RESULTS_FILE), 'RECREATE')
    try:
        for variant in HIST_VARIANTS:
            for pair_key, pair in PAIRS.items():
                directory = out.mkdir(f'{variant}_{pair_key}')
                directory.cd()
                mc_key, data_key = ('mc', pair['mc']), ('data', pair['data'])
                objects = {
                    'hMcInput': RAW[variant].get(mc_key, NORMALIZED[variant][mc_key]),
                    'hDataInput': RAW[variant][data_key],
                    'hMcNorm': NORMALIZED[variant][mc_key], 'hDataNorm': NORMALIZED[variant][data_key],
                    'hMcDensity': vz.density_from_fraction(NORMALIZED[variant][mc_key], 'hMcDensity'),
                    'hDataDensity': vz.density_from_fraction(NORMALIZED[variant][data_key], 'hDataDensity'),
                    'hMcOverData': RATIOS[variant][pair_key]['mc_over_data'],
                    'hDataOverMc': RATIOS[variant][pair_key]['data_over_mc'],
                    'hMcReweightedNorm': REWEIGHTED[variant][pair_key]['norm'],
                    'hClosureRatio': REWEIGHTED[variant][pair_key]['closure_ratio'],
                    'hPulls': FIGURES[(variant, pair_key)]['_pulls'],
                    'fVzWeight': FITS[variant][pair_key].function,
                    'fMcWeight': WEIGHTS[variant][pair_key],
                }
                for order, fit in SCANS[variant][pair_key].items():
                    objects[f'fScanPol{order}'] = fit.function
                for name, obj in objects.items():
                    if obj.Write(name, ROOT.TObject.kOverwrite) <= 0:
                        raise OSError(f'Cannot write {variant}_{pair_key}/{name}')
                ROOT.TNamed('fitSummary', vz.to_json(SUMMARY[f'{variant}/{pair_key}'])).Write('fitSummary')
        directory = out.mkdir(f'{HIST_NAME}_mc_comparison')
        directory.cd()
        for key, entry in MC_COMPARISON.items():
            NORMALIZED[HIST_NAME][('mc', key)].Write(f'hNorm_{key}', ROOT.TObject.kOverwrite)
            if 'ratio' in entry:
                entry['ratio'].Write(f'hRatio_{key}_over_{MC_COMPARISON_REFERENCE}', ROOT.TObject.kOverwrite)
        out.cd()
        ROOT.TNamed('configuration', vz.to_json(dict(
            HIST_NAME=HIST_NAME, MC_TO_DATA_DIRECTION=MC_TO_DATA_DIRECTION,
            MC_COMPARISON_REFERENCE=MC_COMPARISON_REFERENCE, MC_COMPARISON_SAMPLES=MC_COMPARISON_SAMPLES,
            NORMALIZATION_RANGE=NORMALIZATION_RANGE, FIT_RANGE=FIT_RANGE, POLY_ORDER=POLY_ORDER,
            WEIGHT_VALID_RANGE=WEIGHT_VALID_RANGE, DATA_TRIGGER=DATA_TRIGGER, FILE_STEM=FILE_STEM,
            CLOSURE_WEIGHT_MODE=CLOSURE_WEIGHT_MODE))).Write('configuration')
        if out.TestBit(ROOT.TFile.kWriteError):
            raise OSError(f'ROOT write error: {RESULTS_FILE}')
    finally:
        out.Close()

SUMMARY['_mc_comparison'] = dict(reference=MC_COMPARISON_REFERENCE, histogram=HIST_NAME, samples={
    key: {k: v for k, v in entry.items() if k != 'ratio'} for key, entry in MC_COMPARISON.items()})
SUMMARY_JSON = OUTPUT_DIR / 'vz_reweighting_summary.json'
SUMMARY_JSON.write_text(vz.to_json(SUMMARY) + '\n')

SNIPPETS = {}
for variant in HIST_VARIANTS:
    for pair_key, pair in PAIRS.items():
        fit = FITS[variant][pair_key]
        SNIPPETS[(variant, pair_key)] = vz.cpp_snippet(
            fit.parameters, order=POLY_ORDER, fit_range=FIT_RANGE,
            comment_lines=[f'{label_of(pair_key, pair)} MC vs MB data {pair["data"]}, histogram {variant}',
                           f'pol{POLY_ORDER} chi2/ndf = {fit.chi2:.1f}/{fit.ndf}, fit status {fit.status}, '
                           f'cov status {fit.cov_status}'
                           ])
SNIPPET_FILE = OUTPUT_DIR.file('vz_reweighting_fVzWeight.C')  # .file(): '/' treats unknown suffixes as directories
SNIPPET_FILE.write_text('// Generated by hist_analysis/notebooks/02_mc_data_vz_reweighting.ipynb\n'
                        '// Each block gives the pol8 parameters for one MC/data pairing. Production\n'
                        '// holds them per generator and direction in vzWeightParameters()\n'
                        '// (processForestSimple.C); update the matching entry there.\n\n'
                        + '\n'.join(f'// ---- {v} / {p} ----\n{s}' for (v, p), s in SNIPPETS.items()))
print(SNIPPETS[(HIST_NAME, 'embedding_Pbgoing')])
for path in (RESULTS_FILE, SUMMARY_JSON, SNIPPET_FILE):
    print('wrote', path)

In [ ]:
def markdown_summary():
    lines = ['# v_z reweighting summary', '',
             'Generated by `hist_analysis/notebooks/02_mc_data_vz_reweighting.ipynb`.', '',
             '**Convention.** `fVzWeight` is the fitted MC/Data shape ratio f(v_z). '
             'The MC event weight is `1./fVzWeight.Eval(vz)` (divide), exactly as `eventWeight()` in '
             '`processForestSimple.C`. Valid only for |v_z| <= 15 cm (the event selection); never extrapolate.', '',
             f'Data trigger: {DATA_TRIGGER}. Normalization range {NORMALIZATION_RANGE} cm (flow excluded); '
             f'fit range {FIT_RANGE} cm; pol{POLY_ORDER}.', '']
    for variant in HIST_VARIANTS:
        lines += [f'## {variant}', '',
                  '| pairing | MC dir -> data dir | chi2/ndf | max pull | f>0 | closure chi2/ndf before -> after | max abs(r-1) after | max closure pull |',
                  '|---|---|---|---|---|---|---|---|']
        for pair_key, pair in PAIRS.items():
            fit, s, c = FITS[variant][pair_key], STABILITY[variant][pair_key], CLOSURE[variant][pair_key]
            lines.append(f"| {pair_key}{' (diagnostic)' if pair['diagnostic_only'] else ''} | "
                         f"{pair['mc_direction']} -> {pair['data']} | {fit.chi2:.1f}/{fit.ndf} | "
                         f"{s['max_abs_pull']:.1f} | {s['all_positive']} | "
                         f"{c['before']['chi2_ndf']:.1f} -> {c['after']['chi2_ndf']:.2f} | "
                         f"{c['after']['max_abs_ratio_minus_one']:.4f} | {c['after']['max_abs_pull']:.2f} |")
        lines += ['', 'Parameters (p0..p8):', '']
        for pair_key in PAIRS:
            fit = FITS[variant][pair_key]
            lines.append(f"- `{pair_key}`: " + ', '.join(f'{p:.6g}' for p in fit.parameters))
        lines.append('')
    if EVENT_LEVEL:
        lines += ['## Event-level closure (fit on even entries, applied per event to odd entries)', '',
                  '| histogram | sample | chi2/ndf before -> after | max abs(r-1) before -> after |', '|---|---|---|---|']
        for variant, rows in EVENT_LEVEL.items():
            for key, r in rows.items():
                lines.append(f"| {variant} | {key} | {r['before']['chi2_ndf']:.1f} -> {r['after']['chi2_ndf']:.2f} | "
                             f"{r['before']['max_abs_ratio_minus_one']:.3f} -> {r['after']['max_abs_ratio_minus_one']:.4f} |")
    return '\n'.join(lines) + '\n'

SUMMARY_MD = OUTPUT_DIR.file('vz_reweighting_summary.md')
SUMMARY_MD.write_text(markdown_summary())
print(SUMMARY_MD.read_text())

## Automated verification

These checks are numerical, not visual. **Technical** checks must pass, and the cell raises on failure:

* normalization, ratio arithmetic, and error propagation recomputed independently;
* fit convergence, and MINUIT agreeing with the exact least-squares solution;
* the exported file reopens, its TF1s and the parsed C++ snippet reproduce the fitted function, and the event-level provenance holds.

**Physics-quality** criteria are reported as flags against `ACCEPTANCE`, because they describe the data rather than the code:

* $f>0$ over the full range;
* $|$pull$|$ below the threshold;
* the closure improving agreement;
* the closure deviation below the threshold.

In [ ]:
CHECKS = []
def check(name, ok, detail='', technical=True):
    CHECKS.append(dict(name=name, ok=bool(ok), detail=detail, technical=technical))

for variant in HIST_VARIANTS:
    for (kind, key), h in NORMALIZED[variant].items():
        check(f'{variant}/{kind}/{key}: unit normalization', abs(h.Integral(*NORM_BINS) - 1) < 1e-12,
              f'{h.Integral(*NORM_BINS):.15f}')
    for pair_key, pair in PAIRS.items():
        mc = NORMALIZED[variant][('mc', pair['mc'])]; data = NORMALIZED[variant][('data', pair['data'])]
        r = RATIOS[variant][pair_key]; worst_value = worst_error = worst_product = 0.0
        for i in range(NORM_BINS[0], NORM_BINS[1] + 1):
            if i in r['invalid_bins']:
                continue
            a, b, ea, eb = mc.GetBinContent(i), data.GetBinContent(i), mc.GetBinError(i), data.GetBinError(i)
            expected = a / b; expected_error = expected * math.sqrt((ea / a) ** 2 + (eb / b) ** 2)
            worst_value = max(worst_value, abs(r['mc_over_data'].GetBinContent(i) / expected - 1))
            worst_error = max(worst_error, abs(r['mc_over_data'].GetBinError(i) / expected_error - 1))
            worst_product = max(worst_product, abs(r['mc_over_data'].GetBinContent(i) * r['data_over_mc'].GetBinContent(i) - 1))
        check(f'{variant}/{pair_key}: ratio values and errors', max(worst_value, worst_error, worst_product) < 1e-12,
              f'value {worst_value:.1e}, error {worst_error:.1e}, R*W-1 {worst_product:.1e}')
        fit = FITS[variant][pair_key]; s = STABILITY[variant][pair_key]
        check(f'{variant}/{pair_key}: fit status 0 and full covariance', fit.status == 0 and fit.cov_status == 3,
              f'status {fit.status}, cov {fit.cov_status}')
        check(f'{variant}/{pair_key}: MINUIT = exact least squares', fit.max_linear_relative_difference < 1e-5,
              f'{fit.max_linear_relative_difference:.1e}')
        check(f'{variant}/{pair_key}: f > 0 on |vz|<=15', s['all_positive'], f"min {s['min_value']:.4f}", technical=False)
        check(f'{variant}/{pair_key}: max |pull| < {ACCEPTANCE["max_abs_pull"]}', s['max_abs_pull'] < ACCEPTANCE['max_abs_pull'],
              f"{s['max_abs_pull']:.2f}", technical=False)
        c = CLOSURE[variant][pair_key]
        check(f'{variant}/{pair_key}: closure improves chi2', c['after']['chi2'] < c['before']['chi2'],
              f"{c['before']['chi2']:.1f} -> {c['after']['chi2']:.1f}", technical=False)
        check(f'{variant}/{pair_key}: closure max|pull| < {ACCEPTANCE["max_closure_pull"]}',
              c['after']['max_abs_pull'] < ACCEPTANCE['max_closure_pull'],
              f"{c['after']['max_abs_pull']:.2f} sigma (max|r-1| {c['after']['max_abs_ratio_minus_one']:.4f})", technical=False)

# Reopen the export and verify every object and the C++ round trip.
grid = [WEIGHT_VALID_RANGE[0] + 0.01 * k for k in range(int(round((WEIGHT_VALID_RANGE[1] - WEIGHT_VALID_RANGE[0]) / 0.01)) + 1)]
expected_keys = {'hMcInput', 'hDataInput', 'hMcNorm', 'hDataNorm', 'hMcDensity', 'hDataDensity', 'hMcOverData',
                 'hDataOverMc', 'hMcReweightedNorm', 'hClosureRatio', 'hPulls', 'fVzWeight', 'fMcWeight', 'fitSummary'}
with ROOT.TDirectory.TContext():
    reopened = open_root_file(str(RESULTS_FILE))
    try:
        for variant in HIST_VARIANTS:
            for pair_key in PAIRS:
                directory = reopened.GetDirectory(f'{variant}_{pair_key}')
                keys = {k.GetName() for k in directory.GetListOfKeys()} if directory else set()
                check(f'{variant}/{pair_key}: exported keys', expected_keys <= keys, str(sorted(expected_keys - keys)))
                stored = directory.Get('fVzWeight'); weight = directory.Get('fMcWeight')
                fit = FITS[variant][pair_key].function
                parsed = vz.parse_cpp_parameters(SNIPPETS[(variant, pair_key)])
                snippet_function = ROOT.TF1(f'snippet_{variant}_{pair_key}', f'pol{POLY_ORDER}', *FIT_RANGE)
                for k, value in enumerate(parsed):
                    snippet_function.SetParameter(k, value)
                d_stored = max(abs(stored.Eval(x) / fit.Eval(x) - 1) for x in grid)
                d_snippet = max(abs(snippet_function.Eval(x) / fit.Eval(x) - 1) for x in grid)
                d_weight = max(abs(weight.Eval(x) * fit.Eval(x) - 1) for x in grid)
                summary = json.loads(directory.Get('fitSummary').GetTitle())
                check(f'{variant}/{pair_key}: stored TF1, snippet and 1/f reproduce the fit',
                      max(d_stored, d_snippet, d_weight) < 1e-12 and summary['fit']['parameters'] == FITS[variant][pair_key].parameters,
                      f'stored {d_stored:.1e}, snippet {d_snippet:.1e}, weight {d_weight:.1e}')
    finally:
        reopened.Close()

for variant, rows in EVENT_LEVEL.items():
    for key, r in rows.items():
        check(f'event level {key}: selected superset of {variant} (overweight rejection only)',
              0 <= r['selected_excess_fraction'] < 0.03, f"{100*r['selected_excess_fraction']:.2f}%")
        check(f'event level {variant}/{key}: independent closure improves agreement',
              r['after']['chi2'] < r['before']['chi2'], f"{r['before']['chi2_ndf']:.1f} -> {r['after']['chi2_ndf']:.2f}",
              technical=False)
        check(f'event level {variant}/{key}: independent closure max|pull| < {ACCEPTANCE["max_closure_pull"]}',
              r['after']['max_abs_pull'] < ACCEPTANCE['max_closure_pull'], f"{r['after']['max_abs_pull']:.2f} sigma",
              technical=False)

failed_technical = [c for c in CHECKS if c['technical'] and not c['ok']]
flags = [c for c in CHECKS if not c['technical'] and not c['ok']]
print(f'{len(CHECKS)} checks: {sum(c["ok"] for c in CHECKS)} passed, '
      f'{len(failed_technical)} technical failures, {len(flags)} physics-quality flags')
for c in flags:
    print('FLAG  ', c['name'], '|', c['detail'])
for c in failed_technical:
    print('FAIL  ', c['name'], '|', c['detail'])
if failed_technical:
    raise AssertionError(f'{len(failed_technical)} technical checks failed')

### Rerun safety

Every ROOT object created here has a name that includes the variant and pairing. TF1s with an existing name are removed from `gROOT` before they are recreated, and canvases use unique names. The weight closures use fresh Cling namespaces. Re-executing any cell, or the whole notebook, therefore does not collide with earlier objects. The ROOT archive below follows the project convention: rerunning the configuration cell recreates it.

In [ ]:
NOTEBOOK_ROOT_FILE = ROOT_OUTPUT.finish(globals())